# VN-GAT on Colab TPU

**Runtime -> Change runtime type -> TPU** before running anything.

This is the TPU branch. The GPU branch (`scripts/train.py`) is unchanged and
still the reference; nothing here modifies it.


### 1. Install torch_xla


In [ ]:
!pip install -q torch~=2.5.0 torch_xla[tpu]~=2.5.0 \
    -f https://storage.googleapis.com/libtpu-releases/index.html
!pip install -q libigl trimesh

import os
os.environ["PJRT_DEVICE"] = "TPU"

import torch, torch_xla.core.xla_model as xm
print("torch", torch.__version__)
print("xla device:", xm.xla_device(), "| cores:", xm.xrt_world_size())


### 2. Mount Drive

Checkpoints go to Drive so a disconnected runtime does not lose the run.
Colab disconnects far more readily than a Kaggle batch job, so this matters
more here than it did there.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/vngat', exist_ok=True)


### 3. Get the code (TPU branch)


In [ ]:
%cd /content
!git clone -b <TPU_BRANCH_NAME> <YOUR_REPO_URL> vngat
%cd /content/vngat
!python -m scripts.check_version


### 4. Get the data

Breaking Bad is large. Copying it to Drive once and symlinking is usually
faster across sessions than re-downloading, but the first copy is slow.


In [ ]:
%cd /content/vngat
!mkdir -p data

# Option A -- already on Drive:
!ln -sfn /content/drive/MyDrive/breaking_bad/everyday_compressed data/everyday_compressed
!ln -sfn /content/drive/MyDrive/breaking_bad/data_split           data/data_split

# Option B -- download fresh into the (faster, ephemeral) local disk:
# !kaggle datasets download -d dazitu616/breaking-bad-dataset -p /content/bb --unzip
# !ln -sfn /content/bb/everyday_compressed data/everyday_compressed
# !ln -sfn /content/bb/data_split          data/data_split

!python -m scripts.inspect_data --root data | head -20


### 5. Measure the padding cost FIRST

XLA needs static shapes, so batches are padded into buckets. This prints how
many distinct shapes XLA will compile and how much compute the padding wastes.

**Read it before committing a session.** A median edge padding factor of 1.5x
means the TPU must be more than 1.5x faster than the GPU on this workload just
to break even -- and this model is small and scatter-heavy, which is not where
TPUs are strong.


In [ ]:
!python -m scripts.train_tpu --report_buckets --root_dir data \
    --data_subsets everyday_compressed --split_source official


### 6. Sanity checks


In [ ]:
!python -m pytest tests -q


### 7. Train

`batch_size` is **per core**; 8 cores gives an effective batch of 8x it, and
the learning rate is scaled by the core count inside the trainer.

The first few epochs are dominated by XLA compilation -- judge throughput from
epoch 5 onward, not epoch 0.


In [ ]:
!python -m scripts.train_tpu \
    --config configs/colab_tpu.yaml --root_dir data \
    --checkpoint_dir /content/drive/MyDrive/vngat/tpu1 --tag tpu1 --resume none \
    --data_subsets everyday_compressed --split_source official \
    --batch_size 8 --steps_per_epoch 40 --val_steps 8 \
    --epochs 300 --lr 5e-4 --lr_min 5e-5 --num_workers 4


### 8. Resume a later session

`--epochs` is a TOTAL, not an increment. `--restart_schedule true` begins a
fresh anneal at `--lr` over the epochs that remain, instead of inheriting a
rate decided by the ratio of old to new epoch counts.


In [ ]:
!python -m scripts.train_tpu \
    --config configs/colab_tpu.yaml --root_dir data \
    --checkpoint_dir /content/drive/MyDrive/vngat/tpu1 --tag tpu1 --resume auto \
    --epochs 600 --lr 2e-4 --restart_schedule true


### 9. Curves and predictions


In [ ]:
!python -m scripts.plot_history --checkpoint_dir /content/drive/MyDrive/vngat/tpu1 \
    --out /content/curves.png
from IPython.display import Image; Image("/content/curves.png")


In [ ]:
# Checkpoints are saved as CPU tensors, so they evaluate and dump on any backend.
!python -m scripts.evaluate --checkpoint /content/drive/MyDrive/vngat/tpu1/best.pt \
    --split val --num_scenes 100 --by_category --device cpu
